## 관련성 점검 버전
기존 월별 진행 파일을 이어받습니다. 검색어와 정렬은 원본의 치이카와 / relevance를 유지합니다. MAX_SEARCH_CALLS는 실행 전 남은 검색 횟수 이하로 설정하세요.

전체 설명·태그·영상 길이·통계를 50개씩 조회한 후 먼작귀/치이카와/ちいかわ/chiikawa 명칭을 확인합니다. 일치 영상만 videos_df/stats_df/result_df와 checked 분석 CSV에 반영합니다. 명칭 일치가 관련성을 100% 보장하지 않으며 미일치 영상도 검토 대상으로 보존합니다. 기존 데이터 컬럼은 유지하고 상세 설명·태그·판정 근거·수집 시각은 relevance_audit 파일에 별도 저장합니다.

CSV 이름은 youtube_checked_종류_연도.csv입니다. raw는 전체 원본, review는 검토 대상, analysis는 명칭 일치 분석 데이터입니다. 과거 CSV를 덮어쓰지 않습니다. 출력이 비어 있는 범주는 새 파일을 만들지 않으므로 과거 같은 이름 파일은 이번 결과로 오해하지 마세요.

검색 요청 기록은 youtube_search_requests.jsonl에 추가됩니다. 과거 요청은 복원하지 못합니다. 현재까지 데이터를 재검사하려면 검색 셀의 MAX_SEARCH_CALLS=0으로 설정하고 Run All 하세요. 검색 없이 기존 진행 파일을 읽고 videos.list 상세 조회만 수행합니다. 이후 새로운 검색을 이어갈 때는 남은 검색 횟수로 다시 설정하세요. 상세 조회도 쿼터를 사용합니다.


In [ ]:
from pathlib import Path

env_path = Path(".env")
env_path.touch(exist_ok=True)

print(env_path.resolve())

In [ ]:
gitignore_path = Path(".gitignore")
gitignore_content = """
.env
.ipynb_checkpoints/
__pycache__/
"""

gitignore_path.write_text(
    gitignore_content.strip() + "\n",
    encoding="utf-8"
)

print(gitignore_path.resolve())
print(gitignore_path.exists())

In [ ]:
!pip --version

In [ ]:
%pip --version

In [ ]:
%pip install google-api-python-client python-dotenv pandas


In [ ]:
import os
from dotenv import load_dotenv

ENV_PATH = Path.cwd() / ".env"
load_dotenv(ENV_PATH, override=True)

API_KEY = os.getenv("YOUTUBE_API_KEY")

In [ ]:
from googleapiclient.discovery import build


youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

print("Youtube API 서비스 객체 생성")

In [ ]:
# youtube 객체가 사용할 수 있는 메서드 종류

# search() : 영상, 채널, 재생목록 등을 종합적으로 검색
# videos() : 영상에 대한 상세정보 / 통계
# channels() : 유튜브 채널 정보
# playlist() : 유튜브 재생목록 정보
# commentThreads() : 영상 댓글 정보

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
from googleapiclient.errors import HttpError

START_YEAR = 2022
# 실행 전 Cloud Console의 남은 Search Queries per day 이하로 설정하세요.
MAX_SEARCH_CALLS = 20
KEYWORDS = "치이카와"
PROGRESS_FILE = Path("youtube_monthly_progress.json")
KST = timezone(timedelta(hours=9))
REQUEST_LOG = Path("youtube_search_requests.jsonl")

# 최초 실행 시 수집 종료 시점을 고정합니다. 재실행하면 같은 범위를 이어받습니다.
if PROGRESS_FILE.exists():
    progress = json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    if progress["start_year"] != START_YEAR or progress["keywords"] != KEYWORDS:
        raise ValueError("검색 조건이 달라졌습니다. PROGRESS_FILE 이름을 변경하세요.")
else:
    progress = dict(start_year=START_YEAR, keywords=KEYWORDS,
                    end=datetime.now(KST).isoformat(),
                    month=f"{START_YEAR}-01-01T00:00:00+09:00",
                    token=None, tokens=[], items=[], log=[])

def save_progress():
    temp = PROGRESS_FILE.with_suffix(".tmp")
    temp.write_text(json.dumps(progress, ensure_ascii=False), encoding="utf-8")
    temp.replace(PROGRESS_FILE)

items = progress["items"]
seen_ids = {item["id"]["videoId"] for item in items}
end = datetime.fromisoformat(progress["end"])
search_calls = 0

while datetime.fromisoformat(progress["month"]) < end:
    month_start = datetime.fromisoformat(progress["month"])
    next_month = (month_start.replace(year=month_start.year + 1, month=1)
                  if month_start.month == 12 else month_start.replace(month=month_start.month + 1))
    month_end = min(next_month, end)
    if search_calls >= MAX_SEARCH_CALLS:
        print("이번 실행의 검색 횟수 한도 도달. 잔여 쿼터 확인 후 재실행하면 이어받습니다.")
        break
    search_calls += 1
    try:
        search_request = youtube.search().list(
            part="snippet", q=KEYWORDS, type="video", maxResults=50,
            order="relevance", regionCode="KR", relevanceLanguage="ko",
            # 경계에서 정확히 월초에 게시된 영상을 놓치지 않도록 1초 겹칩니다.
            publishedAfter=(month_start - timedelta(seconds=1)).isoformat(),
            publishedBefore=month_end.isoformat(), pageToken=progress["token"]
        )
        # API 키가 들어 있는 요청 URL은 기록하지 않습니다.
        page = search_request.execute()
        record = dict(collected_at=datetime.now(KST).isoformat(),
                      keyword=KEYWORDS, month=month_start.strftime("%Y-%m"),
                      published_after=(month_start - timedelta(seconds=1)).isoformat(),
                      published_before=month_end.isoformat(), order="relevance",
                      page_token=progress["token"], next_page_token=page.get("nextPageToken"),
                      returned=len(page.get("items", [])),
                      video_ids=[x.get("id", {}).get("videoId") for x in page.get("items", [])],
                      titles=[x.get("snippet", {}).get("title", "") for x in page.get("items", [])])
        with REQUEST_LOG.open("a", encoding="utf-8") as log_file:
            log_file.write(json.dumps(record, ensure_ascii=False) + "\n")
    except HttpError as error:
        print(f"검색 중단: HTTP {error.resp.status}. 진행 상황은 보존됩니다.")
        break

    for item in page.get("items", []):
        video_id = item.get("id", {}).get("videoId")
        published = item.get("snippet", {}).get("publishedAt")
        if not video_id or not published:
            continue
        published = datetime.fromisoformat(published.replace("Z", "+00:00"))
        if month_start <= published < month_end and video_id not in seen_ids:
            seen_ids.add(video_id)
            items.append(item)

    token = page.get("nextPageToken")
    reason = ("empty_page" if not page.get("items") else
              "no_next_token" if not token else
              "repeated_token" if token in progress["tokens"] else None)
    if reason:
        progress["log"].append(dict(month=month_start.strftime("%Y-%m"), reason=reason))
        print(f"{month_start:%Y-%m} 검색 종료 ({reason}) / 누적 고유 영상 {len(items)}개")
        progress.update(month=next_month.isoformat(), token=None, tokens=[])
    else:
        progress["token"] = token
        progress["tokens"].append(token)
    save_progress()

response = {"items": items}
print(f"이번 실행 검색 {search_calls}회 / 누적 고유 영상 {len(items)}개")
if datetime.fromisoformat(progress["month"]) >= end:
    print(f"전체 기간 검색 완료: {end:%Y-%m-%d %H:%M} (한국시간)까지")

# 기존 컬럼을 유지하면서 한국시간 게시 연도별로 CSV를 분리합니다.
def save_yearly_csv(frame, name):
    if frame.empty:
        print(f"{name}: 저장할 데이터가 없습니다.")
        return
    dates = {item["id"]["videoId"]: datetime.fromisoformat(
        item["snippet"]["publishedAt"].replace("Z", "+00:00")
    ).astimezone(KST).year for item in items}
    years = frame["video_id"].map(dates)
    for year in sorted(years.dropna().unique()):
        filename = f"youtube_checked_{name}_{int(year)}.csv"
        frame.loc[years == year].to_csv(filename, index=False, encoding="utf-8-sig")
        print(f"저장: {filename}")


In [ ]:
response

In [ ]:
# -------------------------------------------------------------
# 채널 ID, 채널명, 게시일시, 게시물 ID, 게시물 제목, 게시물 본문
# -------------------------------------------------------------

import html
import pandas as pd

videos = []

for item in response["items"] :

    snippet = item["snippet"] 

    videos.append({
        "channel_id" : snippet["channelId"], # 채널ID
        "channel" : html.unescape(snippet["channelTitle"]), # 채널명
        "publish_at" : snippet["publishedAt"], # 게시일시
        "video_id" : item["id"]["videoId"], # 게시물ID
        "title" : html.unescape(snippet["title"]), # 게시물 제목
        "description" : html.unescape(snippet["description"]) # 게시물 본문
    })

videos_df = pd.DataFrame(videos, columns=["channel_id", "channel", "publish_at", "video_id", "title", "description"])
videos_df.head()


In [ ]:
# 관련성 검사 후 아래에서 연도별로 저장합니다.


In [ ]:
# videos_df의 고유 영상 ID를 50개씩 조회합니다.
video_ids = videos_df["video_id"].dropna().drop_duplicates().tolist()

In [ ]:
video_ids

In [ ]:
stats_response = {"items": []}
detail_items = []
details_collected_at = datetime.now(KST).isoformat()

for start in range(0, len(video_ids), 50):
    try:
        stats_request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_ids[start:start + 50])
        )
        batch = stats_request.execute()
    except HttpError as error:
        print(f"통계 조회 중단 (일부 결과만 포함): HTTP {error.resp.status}")
        break

    stats_response["items"].extend(batch.get("items", []))
    detail_items.extend(batch.get("items", []))

print(f"통계 {len(stats_response['items'])}개 조회 완료")


In [ ]:
stats_response

In [ ]:
# -------------------------------------------------------------
# 게시물 ID, 조회수, 좋아요수, 댓글수
# -------------------------------------------------------------

statistics = []

for item in stats_response["items"] :
    stats = item.get("statistics", {})

    statistics.append({
        "video_id": item['id'],
        "view_count" : stats.get("viewCount"),
        "like_count" : stats.get("likeCount"),
        "comment_count" : stats.get("commentCount")
    })

stats_df = pd.DataFrame(statistics, columns=["video_id", "view_count", "like_count", "comment_count"])

In [ ]:
stats_df.head()

In [ ]:
# 통계도 관련성 검사 후 저장합니다.


In [ ]:
result_df = pd.merge(videos_df, stats_df, on="video_id", how="left")

In [ ]:
result_df.info()

In [ ]:
cols = ["view_count", "like_count", "comment_count"]

for col in cols :
    result_df[col] = pd.to_numeric(
        result_df[col],
        errors="coerce"
    )

result_df.info()

In [ ]:
# 다음 셀에서 관련 영상과 검토 대상을 분리해 저장합니다.


In [ ]:
import re
import unicodedata

# 명칭이 명시된 영상을 분석 대상으로 삼는 재현 가능한 기준입니다.
# 불일치는 무관한 영상으로 확정하지 않고 검토 대상으로 보존합니다.
pattern = re.compile(r"먼작귀|치이카와|ちいかわ|chiikawa", re.IGNORECASE)

def normalize_text(value):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", value or ""))

details_by_id = {item["id"]: item for item in detail_items}
audit_rows = []
for row in pd.DataFrame(videos, columns=["channel_id", "channel", "publish_at", "video_id", "title", "description"]).to_dict("records"):
    detail = details_by_id.get(row["video_id"])
    snippet = detail.get("snippet", {}) if detail else {}
    title = html.unescape(snippet.get("title", row["title"]))
    description = html.unescape(snippet.get("description", row["description"]))
    tags = snippet.get("tags", [])
    fields = {"title": title, "description": description, "tags": " ".join(tags)}
    matches = [name for name, value in fields.items() if pattern.search(normalize_text(value))]
    # 상세 조회에 실패한 영상은 검색 문구가 일치해도 검토 대상으로 둡니다.
    status = ("relevant" if matches else "review_no_match") if detail else "review_details_unavailable"
    audit_rows.append(dict(video_id=row["video_id"],
                           relevance_status=status, matched_fields="|".join(matches),
                           full_title=title, full_description=description,
                           tags="|".join(tags),
                           duration=detail.get("contentDetails", {}).get("duration") if detail else None,
                           collected_at=details_collected_at))

audit_df = pd.DataFrame(audit_rows, columns=[
    "video_id", "relevance_status", "matched_fields", "full_title",
    "full_description", "tags", "duration", "collected_at"
])

# 원본 데이터와 기존 컬럼을 보존하고 분석 대상만 별도로 저장합니다.
all_videos_df = pd.DataFrame(videos, columns=["channel_id", "channel", "publish_at", "video_id", "title", "description"])
all_stats_df = pd.DataFrame(statistics, columns=["video_id", "view_count", "like_count", "comment_count"])
all_result_df = pd.merge(all_videos_df, all_stats_df, on="video_id", how="left")
for col in ["view_count", "like_count", "comment_count"]:
    all_result_df[col] = pd.to_numeric(all_result_df[col], errors="coerce")
relevant_ids = set(audit_df.loc[audit_df["relevance_status"] == "relevant", "video_id"])
relevant_df = all_videos_df.loc[all_videos_df["video_id"].isin(relevant_ids)].copy()
review_df = all_videos_df.loc[~all_videos_df["video_id"].isin(relevant_ids)].copy()

videos_df = relevant_df.copy()
stats_df = all_stats_df.loc[all_stats_df["video_id"].isin(relevant_ids)].copy()
result_df = all_result_df.loc[all_result_df["video_id"].isin(relevant_ids)].copy()

save_yearly_csv(all_videos_df, "raw_videos")
save_yearly_csv(all_result_df, "raw_analysis")
save_yearly_csv(videos_df, "videos")
save_yearly_csv(stats_df, "stats")
save_yearly_csv(result_df, "analysis")
save_yearly_csv(review_df, "review")
save_yearly_csv(audit_df, "relevance_audit")
print(f"전체: {len(all_videos_df)} / 관련 명칭 일치: {len(videos_df)} / 검토 대상: {len(review_df)}")


In [ ]:
# -------------------------------------------------------------
# 댓글 정보 수집
# -------------------------------------------------------------

In [ ]:
# # 특정 영상의 video_id 추출
# video_id = result_df.loc[0, "video_id"]

# # 댓글 수집
# request = youtube.commentThreads().list(
#     part="snippet",
#     videoId = video_id,
#     maxResults=100,
#     order="relevance",
#     textFormat="plainText"
# )

# response = request.execute()
# response

part(= 데이터 영역 선택)값 종류

> 모든 요소의 part가 동일x
> 특정 메서드만의 part 값도 존재

- snippet : 제목, 설명, 채널명, 태그, 게시일 등 정보
- statistics : 조회수, 좋아요수, 댓글수
- contentDetails : 영상길이, 화질 정보
- topicDetails : 영상 관련 주제
- recodingDetails : 녹화일시
- liveStreamingDetails : 라이브 방송 시작, 종료 시간
- localizations : 언어별 제목과 설명
- replies : 댓글에 달린 대댓글

In [ ]:
# from googleapiclient.errors import HttpError

# video_ids = result_df["video_id"].dropna().unique().tolist()

# comments = []

# for video_id in video_ids:

#     try:
#         request = youtube.commentThreads().list(
#             part="snippet",
#             videoId=video_id,
#             maxResults=100,
#             order="relevance",
#             textFormat="plainText"
#         )

#         response = request.execute()


#         for item in response.get("items", []) :

#             snippet = item["snippet"]
#             comment = snippet["topLevelComment"]["snippet"]

#             comments.append({
#                 "video_id": video_id,
#                 "comment_id": snippet["topLevelComment"]["id"],
#                 "author": comment.get("authorDisplayName"),
#                 "text": comment.get("textDisplay"),
#                 "like_count":comment.get("likeCount"),
#                 "published_at":comment.get("publishedAt"),
#                 "reply_count":snippet.get("totalReplyCount")
#             })

#         print(f"{video_id}: 댓글 수집 완료")

#     except HttpError as e :
#         print(f"{video_id}: 댓글 수집 실패 - {e}")


# comment_df = pd.DataFrame(comments)

# comment_df.head()

In [ ]:
# # 1개 영상당 최대 1000개까지 댓글을 가져오려고 할 때

# from googleapiclient.errors import HttpError

# video_ids = result_df["video_id"].dropna().unique().tolist()

# comments = []

# TARGET_COUNT = 1000 # 영상 1개당 최대 수집할 댓글 개수
# PAGE_SIZE = 100 # API 1회 요청당 최대 댓글 수

# for video_id in video_ids:

#     page_token = None # 모든 페이지가 nextpagetoken 값이 있진 않으므로 default값으로 None
#     collected_count = 0

#     print(f"영상 {video_id} 댓글 수집 시작")

#     while collected_count < TARGET_COUNT :

#         try:
#             request = youtube.commentThreads().list(
#                 part="snippet",
#                 videoId=video_id,
#                 maxResults=PAGE_SIZE,
#                 order="relevance",
#                 textFormat="plainText",
#                 pageToken=page_token
#             )

#             response = request.execute()

#         except HttpError as e :
#             print(f"{video_id}: 댓글 수집 실패 - {e}")


#         for item in response.get("items", []) :

#             snippet = item["snippet"]
#             comment = snippet["topLevelComment"]["snippet"]

#             comments.append({
#                 "video_id": video_id,
#                 "comment_id": snippet["topLevelComment"]["id"],
#                 "author": comment.get("authorDisplayName"),
#                 "text": comment.get("textDisplay"),
#                 "like_count":comment.get("likeCount"),
#                 "published_at":comment.get("publishedAt"),
#                 "reply_count":snippet.get("totalReplyCount")
#             })

#             collected_count += 1

#             if collected_count >= TARGET_COUNT :
#                 break

#         print(f"{video_id}: 댓글 수집 완료")

#         page_token = response.get("nextPageToken")

#         if not page_token or not item :
#             break

#     print(f"영상 {video_id} 최종 수집 : {collected_count}개")


# comment_df = pd.DataFrame(comments)

# comment_df.head()